# 🩺 BMI Categorizer & Diabetes Predictor
## Jupyter Notebook — Model Training

**Dataset:** Pima Indians Diabetes Dataset  
**Algorithm:** Logistic Regression  
**Goal:** Train a model and save it as `model.pkl` for the Flask app

---

## Step 1: Import Libraries
We import all the tools we need.

In [ ]:
# pandas  → for loading and manipulating data (like Excel for Python)
# numpy   → for numerical calculations
# matplotlib & seaborn → for creating graphs and charts
# sklearn → machine learning tools
# pickle  → to save our trained model to a file

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, roc_auc_score
)
import pickle

print('✅ All libraries imported successfully!')

## Step 2: Load the Dataset

**About the Pima Indians Diabetes Dataset**

| Column | Meaning |
|---|---|
| Pregnancies | Number of times pregnant |
| Glucose | Plasma glucose concentration (mg/dL) |
| BloodPressure | Diastolic blood pressure (mm Hg) |
| SkinThickness | Triceps skin fold thickness (mm) |
| Insulin | 2-Hour serum insulin (μU/mL) |
| BMI | Body mass index (weight in kg/height in m²) |
| DiabetesPedigreeFunction | Family history likelihood score |
| Age | Age in years |
| Outcome | 0 = Not Diabetic, 1 = Diabetic |

📥 **Download from:** https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database

In [ ]:
# Load the CSV file into a DataFrame (like a table/spreadsheet)
df = pd.read_csv('../diabetes.csv')

# Show the first 5 rows
print(f'Dataset shape: {df.shape}')   # (rows, columns)
df.head()

In [ ]:
# Check basic statistics (min, max, mean, etc.)
df.describe()

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())
print()

# Check class balance
print('Outcome distribution:')
print(df['Outcome'].value_counts())
print(f"  0 = Not Diabetic: {(df['Outcome']==0).sum()}")
print(f"  1 = Diabetic:     {(df['Outcome']==1).sum()}")

## Step 3: Exploratory Data Analysis (EDA)
Let's visualize the data to understand it better.

In [ ]:
# Pie chart — how many diabetic vs not diabetic
plt.figure(figsize=(5, 5))
df['Outcome'].value_counts().plot.pie(
    labels=['Not Diabetic', 'Diabetic'],
    autopct='%1.1f%%',
    colors=['#4ade80', '#f87171'],
    startangle=90
)
plt.title('Diabetes Distribution', fontsize=14)
plt.ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of each feature
features = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
            'Insulin','BMI','DiabetesPedigreeFunction','Age']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(features):
    axes[i].hist(df[col], bins=25, color='#3dd6cc', edgecolor='#0d1826', alpha=0.8)
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

plt.suptitle('Feature Distributions', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap — which features are related to each other?
plt.figure(figsize=(10, 8))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

## Step 4: Data Preprocessing

Some columns have `0` values that are **medically impossible** (e.g., Glucose = 0).  
These represent **missing values** — we replace them with the **median** of the column.

In [ ]:
# These columns cannot logically be zero — treat 0 as missing
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

print('Zero counts before cleaning:')
for col in zero_cols:
    print(f'  {col}: {(df[col] == 0).sum()} zeros')

# Replace 0 with NaN, then fill NaN with the column median
for col in zero_cols:
    df[col] = df[col].replace(0, np.nan)
    df[col] = df[col].fillna(df[col].median())

print('\n✅ Zeros replaced with column medians.')
print('Missing values now:', df.isnull().sum().sum())

## Step 5: Split Data into Features (X) and Target (y)

In [ ]:
# X = input features (everything except Outcome)
# y = what we want to predict (Outcome: 0 or 1)

X = df.drop('Outcome', axis=1)   # all columns except Outcome
y = df['Outcome']                 # only the Outcome column

print(f'X shape (features): {X.shape}')   # (rows, 8 features)
print(f'y shape (target):   {y.shape}')   # (rows,)
print(f'\nFeature columns: {list(X.columns)}')

In [ ]:
# Split into training set (80%) and test set (20%)
# The model LEARNS from training data and is TESTED on test data
# stratify=y ensures both splits have similar Diabetic/Not Diabetic ratio

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% for testing
    random_state=42,     # fixed seed for reproducibility
    stratify=y           # keep class balance
)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples:  {X_test.shape[0]}')

## Step 6: Feature Scaling

Logistic Regression works better when all features are on the **same scale**.  
`StandardScaler` converts each feature to have mean=0 and std=1.

In [ ]:
# Create a scaler object
scaler = StandardScaler()

# Fit on TRAINING data only, then transform both sets
# (never fit on test data — that would be cheating!)
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('✅ Features scaled.')
print(f'Mean of Glucose after scaling: {X_train_sc[:, 1].mean():.4f}  (should be ≈ 0)')
print(f'Std  of Glucose after scaling: {X_train_sc[:, 1].std():.4f}   (should be ≈ 1)')

## Step 7: Train Logistic Regression Model

**Why Logistic Regression?**
- Simple and interpretable
- Works well for binary classification (Diabetic / Not Diabetic)
- Outputs a probability — so we can show confidence %
- Low risk of overfitting on small datasets

In [ ]:
# Create the model
model = LogisticRegression(
    max_iter=1000,   # number of iterations to converge
    random_state=42  # reproducibility
)

# Train the model — it learns the pattern in the training data
model.fit(X_train_sc, y_train)

print('✅ Model trained successfully!')

## Step 8: Evaluate the Model

In [ ]:
# Make predictions on the TEST set
y_pred = model.predict(X_test_sc)

# Accuracy = (correct predictions) / (total predictions) × 100
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, model.predict_proba(X_test_sc)[:, 1])

print(f'📊 Accuracy:  {acc * 100:.2f}%')
print(f'📊 ROC-AUC:   {auc:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Not Diabetic', 'Diabetic']))

In [ ]:
# Confusion Matrix
# Shows: True Positives, True Negatives, False Positives, False Negatives
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Diabetic', 'Diabetic'],
            yticklabels=['Not Diabetic', 'Diabetic'])
plt.title('Confusion Matrix', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance — which features matter most?
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values('Coefficient', ascending=True)

plt.figure(figsize=(8, 5))
colors = ['#f87171' if c > 0 else '#4ade80' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.axvline(0, color='white', linewidth=0.8)
plt.title('Feature Coefficients (Logistic Regression)', fontsize=13)
plt.xlabel('Coefficient Value')
plt.tight_layout()
plt.show()
print('Red bars → increase diabetes risk | Green bars → decrease risk')

## Step 9: Save Model to model.pkl

In [ ]:
# Save BOTH the model and the scaler
# The Flask app needs both to make predictions

with open('../model.pkl', 'wb') as f:
    pickle.dump({'model': model, 'scaler': scaler}, f)

print('✅ model.pkl saved to ML_Project/')
print('The Flask app (app.py) will load this file automatically.')

## ✅ Done!

Now go back to your terminal and run:
```bash
python app.py
```
Then open **http://127.0.0.1:5000** in your browser.